In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.prompts import PromptTemplate

### RAG Pipeline

In [7]:
# load doc:
docs = PyPDFLoader('../data/SDE_Resume_Sandip.pdf').load()

# Split docs in chunks:
split_docs = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)

# vector embbedings:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

# store in vector DB:
vector_store = Chroma.from_documents(
    documents=split_docs,
    embedding=embeddings,
    persist_directory="./vector_db_02"
)

In [ ]:
llm = ChatGroq(
    model="qwen/qwen3.6-27b",
)


### Chaining | get context -> prompt -> llm -> output

In [16]:
# Get context
def get_context(query: str):
    res = vector_store.similarity_search(query=query, k=3)

    context = "\n\n".join(doc.page_content for doc in res)

    return {
        "context": context,
        "query": query
    }


# Prompt
prompt = PromptTemplate(
    template="""
You are a helpful assistant. Answer the user's query based only on the provided context.

Context:{context}
Query:{query}
""",
    input_variables=["context", "query"]
)


# RAG chain
rag_chain = get_context | prompt | llm


# Invoke
query = "What is GenAI?"

res = rag_chain.invoke(query)

print(res.content)


<think>
Here's a thinking process:

1.  **Analyze User Query:** The user asks "What is GenAI?"
2.  **Analyze Context:** The provided context contains two main sections:
   - A section about cricket ("The Changing Nature of Cricket", "Technology and Modern T20 Cricket", "Global Growth of Cricket")
   - A resume/CV for "Sandip Sarkar" detailing technical skills, projects, and experience.
   - I need to scan the context for any mention of "GenAI", "Generative AI", or related terms.
   - Scanning... I see mentions of "AI-powered", "LLM APIs", "Google Gemini", "AI-driven interfaces", but no explicit definition or explanation of "GenAI".
3.  **Evaluate Findings:** The context does not contain any information defining or explaining what GenAI is. It only mentions AI in the context of projects and skills (e.g., "AI-powered SaaS platform", "LLM APIs", "Google Gemini").
4.  **Formulate Response:** Since the instruction strictly says "Answer the user's query based only on the provided context," 